## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [4]:
## Parameter 
pre_start_date = '20240201'
pre_end_date = '20240220'

post_start_date = '20240301'
post_end_date = '20240320'

## PRE Data

In [10]:
## base

pre_base_query = f"""

with experiment_user_base as (
    
    select
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{pre_start_date}'
        and yyyymmdd <= '{pre_end_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{pre_start_date}'
            and yyyymmdd <= '{pre_end_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        -- city,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1
    order by 1

"""

df_pre_agg = pd.read_sql(pre_base_query, connection)
df_pre_agg.head(5)

,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,l1Cancel,20240201 to 20240220,97469,1129131,649982,57.56,18.09,0.1,294,1509826,14088,0.93
1,l2Cancel,20240201 to 20240220,94905,1111315,635285,57.17,18.15,0.1,286,1478043,13711,0.93


In [14]:
## base

pre_city_query = f"""

with experiment_user_base as (
    
    select
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{pre_start_date}'
        and yyyymmdd <= '{pre_end_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{pre_start_date}'
            and yyyymmdd <= '{pre_end_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        city,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1,2
    order by 1,2

"""

df_pre_city = pd.read_sql(pre_city_query, connection)
df_pre_city.head(5)

,city,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Bangalore,l1Cancel,20240201 to 20240220,43717,531581,283498,53.33,16.42,0.10,273,674660,5466,0.81
1,Bangalore,l2Cancel,20240201 to 20240220,42770,528841,280514,53.04,16.33,0.10,267,666877,5171,0.78
2,Delhi,l1Cancel,20240201 to 20240220,42357,492581,303756,61.67,18.99,0.09,307,691832,6950,1.00
3,Delhi,l2Cancel,20240201 to 20240220,41257,479809,293353,61.14,19.24,0.10,303,671550,6866,1.02
4,Jaipur,l1Cancel,20240201 to 20240220,5244,39958,21581,54.01,20.65,0.10,354,52016,468,0.90


In [18]:
## base

pre_service_query = f"""

with experiment_user_base as (
    
    select
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{pre_start_date}'
        and yyyymmdd <= '{pre_end_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{pre_start_date}'
            and yyyymmdd <= '{pre_end_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        service_name service,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1,2
    order by 1,2

"""

df_pre_service = pd.read_sql(pre_service_query, connection)
df_pre_service.head(5)

,service,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Auto,l1Cancel,20240201 to 20240220,57513,381993,195913,51.29,16.80,0.08,255,619098,5209,0.84
1,Auto,l2Cancel,20240201 to 20240220,56175,376535,192338,51.08,16.79,0.08,257,608162,4885,0.80
2,Bike Lite,l1Cancel,20240201 to 20240220,12998,76826,52866,68.81,17.11,0.07,327,151171,1461,0.97
3,Bike Lite,l2Cancel,20240201 to 20240220,12982,74709,51263,68.62,17.14,0.07,320,147415,1387,0.94
4,Bike Metro,l1Cancel,20240201 to 20240220,2941,14202,8915,62.77,18.74,0.04,309,30253,281,0.93


## POST Data

In [11]:
## base

post_base_query = f"""

with experiment_user_base as (
    
    select
        yyyymmdd,
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1,2
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{post_start_date}'
        and yyyymmdd <= '{post_start_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.yyyymmdd = a.yyyymmdd
        AND exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.yyyymmdd = exp_base.yyyymmdd
        and a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        -- city,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1
    order by 1

"""

df_post_agg = pd.read_sql(post_base_query, connection)
df_post_agg.head(5)  

,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,l1Cancel,20240301 to 20240301,128541,225891,139670,61.83,21.05,0.15,327,354463,5110,1.44
1,l2Cancel,20240301 to 20240301,127088,220681,138622,62.82,20.24,0.17,337,354762,5112,1.44


In [15]:
## base

post_city_query = f"""

with experiment_user_base as (
    
    select
        yyyymmdd,
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1,2
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{post_start_date}'
        and yyyymmdd <= '{post_start_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.yyyymmdd = a.yyyymmdd
        AND exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.yyyymmdd = exp_base.yyyymmdd
        and a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        city,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1,2
    order by 1,2

"""

df_post_city = pd.read_sql(post_city_query, connection)
df_post_city.head(5)  

,city,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Bangalore,l1Cancel,20240301 to 20240301,55163,101906,62015,60.86,18.68,0.14,301,162804,2453,1.51
1,Bangalore,l2Cancel,20240301 to 20240301,54678,99658,61722,61.93,17.87,0.15,317,163179,2474,1.52
2,Delhi,l1Cancel,20240301 to 20240301,54999,93117,59963,64.40,21.95,0.15,345,145785,1879,1.29
3,Delhi,l2Cancel,20240301 to 20240301,54406,91225,59446,65.16,21.18,0.17,359,145679,1871,1.28
4,Jaipur,l1Cancel,20240301 to 20240301,8171,14363,7671,53.41,23.51,0.18,384,21112,332,1.57


In [19]:
## base

post_service_query = f"""

with experiment_user_base as (
    
    select
        yyyymmdd,
        userId,
        max(pageVariant) pageVariant
    from
        (
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_ontheway_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_arrived_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        
        union
        
        select 
            yyyymmdd,
            json_extract_scalar(event_props,'$.userId') userId,
            json_extract_scalar(event_props,'$.pageVariant') pageVariant
        from 
            clevertap.customer_postorderstatus_started_immutable
        where 
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            and json_extract_scalar(event_props,'$.pageVariant') IN ('l1Cancel', 'l2Cancel')
            and json_extract_scalar(event_props,'$.currentCity') in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        )
    group by 1,2
    ), 
    
    orderLogsFact as (

    select
        yyyymmdd,
        case 
        when city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata') then city_name
        else 'other' end city,
        service_obj_service_name service_name,
        customer_id,
        order_id,
        order_status,
        modified_order_status,
        spd_fraud_flag,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
    from
        orders.order_logs_fact
    where 
        yyyymmdd >= '{post_start_date}'
        and yyyymmdd <= '{post_start_date}'
        and city_name in ('Bangalore', 'Delhi', 'Jaipur', 'Kolkata')
        and service_obj_service_name in ('Link', 'Auto', 'Bike Lite', 'Bike Metro', 'CabEconomy')
    ),
    
    banner_monetisation AS (
    
    SELECT 
        a.yyyymmdd,
        user_id,
        ct_session_id,
        order_status,
        MAX(CASE WHEN action = 'view' THEN unique_id END) view,
        MAX(CASE WHEN action = 'click' THEN unique_id END) click
    FROM
        (
        SELECT
            yyyymmdd,
            event_props_user_id user_id,
            event_props_action action,
            event_props_order_status || 'Screen' order_status,
            cast(cast(event_props_ct_session_id as decimal) as varchar) ct_session_id,
            event_props_user_id || '-' || event_props_action || '-' || hhmmss unique_id
        FROM    
            clevertap.customer_banner_monetisation_immutable
        WHERE  
            yyyymmdd >= '{post_start_date}'
            and yyyymmdd <= '{post_start_date}'
            -- AND event_props_current_city IN (SELECT city_display_name FROM service_mapping)
            AND event_props_action IN ('view', 'click')
            -- AND event_props_type = 'GAMBanner'
            AND event_props_order_status in ('onTheWay', 'arrived', 'started')
            
        GROUP BY 1,2,3,4,5,6
        ) AS a
    JOIN 
        experiment_user_base exp
        ON exp.yyyymmdd = a.yyyymmdd
        AND exp.userId = a.user_id
        
    GROUP BY 1,2,3,4
    ),
    
    merge as (
    
    select
        a.yyyymmdd,
        a.city,
        a.service_name,
        exp_base.pageVariant,
        a.customer_id,
        a.order_id,
        a.order_status,
        a.spd_fraud_flag,
        a.modified_order_status,
        a.time_spent_in_sec,
        bm.order_status postorderstatus,
        view,
        click
    from 
        orderLogsFact a 
    join 
        experiment_user_base exp_base
        on a.yyyymmdd = exp_base.yyyymmdd
        and a.customer_id = exp_base.userId
    left join 
        banner_monetisation bm
        on a.yyyymmdd = bm.yyyymmdd
        and a.customer_id = bm.user_id
    -- where 
    --     order_id = '65d600782455246680859d29'
    )

    select
        -- yyyymmdd,
        service_name service,
        pageVariant,
        min(yyyymmdd) || ' to ' || max(yyyymmdd) date_range,
        count(distinct customer_id) customers,
        count(distinct order_id) gross_orders,
        count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end) net_orders,
        coalesce(try(count(distinct case when order_status = 'dropped' and spd_fraud_flag != true then order_id end)*100.00/count(distinct order_id)),0)  g2n,
        coalesce(try(count(distinct case when modified_order_status = 'OCARA' then order_id end)*100.00/count(distinct order_id)),0)  ocara,
        coalesce(try(count(distinct case when modified_order_status = 'aborted' then order_id end)*100.00/count(distinct order_id)),0)  aborted,
        approx_percentile(CASE WHEN modified_order_status = 'OCARA' THEN time_spent_in_sec END,0.50) as median_ocara_time,
            
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end) view,
        count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end) click,
        coalesce(try(count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then click end)*100.00/count(distinct case when postorderstatus in ('onTheWayScreen', 'arrivedScreen', 'startedScreen') then view end)), 0) ctr
        
    from 
        merge 

    group by 1,2
    order by 1,2

"""

df_post_service = pd.read_sql(post_service_query, connection)
df_post_service.head(5)  

,service,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Auto,l1Cancel,20240301 to 20240301,47636,77363,45593,58.93,19.01,0.12,285,146602,2026,1.38
1,Auto,l2Cancel,20240301 to 20240301,46929,75599,45385,60.03,18.30,0.14,302,146534,2013,1.37
2,Bike Lite,l1Cancel,20240301 to 20240301,14408,21131,14665,69.40,18.23,0.09,337,41227,521,1.26
3,Bike Lite,l2Cancel,20240301 to 20240301,13852,20455,14208,69.46,18.23,0.08,357,39801,535,1.34
4,Bike Metro,l1Cancel,20240301 to 20240301,1575,2006,1353,67.45,21.14,0.15,408,4805,47,0.98


## Analysis

In [23]:
df_pre_agg

,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,l1Cancel,20240201 to 20240220,97469,1129131,649982,57.56,18.09,0.1,294,1509826,14088,0.93
1,l2Cancel,20240201 to 20240220,94905,1111315,635285,57.17,18.15,0.1,286,1478043,13711,0.93


In [25]:
df_post_agg

,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,l1Cancel,20240301 to 20240301,128541,225891,139670,61.83,21.05,0.15,327,354463,5110,1.44
1,l2Cancel,20240301 to 20240301,127088,220681,138622,62.82,20.24,0.17,337,354762,5112,1.44


In [27]:
df_pre_city

,city,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Bangalore,l1Cancel,20240201 to 20240220,43717,531581,283498,53.33,16.42,0.10,273,674660,5466,0.81
1,Bangalore,l2Cancel,20240201 to 20240220,42770,528841,280514,53.04,16.33,0.10,267,666877,5171,0.78
2,Delhi,l1Cancel,20240201 to 20240220,42357,492581,303756,61.67,18.99,0.09,307,691832,6950,1.00
3,Delhi,l2Cancel,20240201 to 20240220,41257,479809,293353,61.14,19.24,0.10,303,671550,6866,1.02
4,Jaipur,l1Cancel,20240201 to 20240220,5244,39958,21581,54.01,20.65,0.10,354,52016,468,0.90
5,Jaipur,l2Cancel,20240201 to 20240220,4982,38841,21084,54.28,20.29,0.12,338,49926,410,0.82
6,Kolkata,l1Cancel,20240201 to 20240220,7047,65011,41147,63.29,23.40,0.15,290,93024,1226,1.32
7,Kolkata,l2Cancel,20240201 to 20240220,6766,63824,40334,63.20,23.72,0.16,300,90992,1283,1.41


In [29]:
df_post_city

,city,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Bangalore,l1Cancel,20240301 to 20240301,55163,101906,62015,60.86,18.68,0.14,301,162804,2453,1.51
1,Bangalore,l2Cancel,20240301 to 20240301,54678,99658,61722,61.93,17.87,0.15,317,163179,2474,1.52
2,Delhi,l1Cancel,20240301 to 20240301,54999,93117,59963,64.40,21.95,0.15,345,145785,1879,1.29
3,Delhi,l2Cancel,20240301 to 20240301,54406,91225,59446,65.16,21.18,0.17,359,145679,1871,1.28
4,Jaipur,l1Cancel,20240301 to 20240301,8171,14363,7671,53.41,23.51,0.18,384,21112,332,1.57
5,Jaipur,l2Cancel,20240301 to 20240301,7962,13742,7490,54.50,22.98,0.16,427,20950,366,1.75
6,Kolkata,l1Cancel,20240301 to 20240301,10287,16505,10021,60.71,28.37,0.20,343,25248,465,1.84
7,Kolkata,l2Cancel,20240301 to 20240301,10119,16056,9964,62.06,27.32,0.24,393,25397,411,1.62


In [31]:
df_pre_service

,service,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Auto,l1Cancel,20240201 to 20240220,57513,381993,195913,51.29,16.80,0.08,255,619098,5209,0.84
1,Auto,l2Cancel,20240201 to 20240220,56175,376535,192338,51.08,16.79,0.08,257,608162,4885,0.80
2,Bike Lite,l1Cancel,20240201 to 20240220,12998,76826,52866,68.81,17.11,0.07,327,151171,1461,0.97
3,Bike Lite,l2Cancel,20240201 to 20240220,12982,74709,51263,68.62,17.14,0.07,320,147415,1387,0.94
4,Bike Metro,l1Cancel,20240201 to 20240220,2941,14202,8915,62.77,18.74,0.04,309,30253,281,0.93
5,Bike Metro,l2Cancel,20240201 to 20240220,2801,14071,8529,60.61,19.10,0.08,304,28728,294,1.02
6,CabEconomy,l1Cancel,20240201 to 20240220,26603,99249,51374,51.76,23.94,0.12,328,203101,1839,0.91
7,CabEconomy,l2Cancel,20240201 to 20240220,26072,97446,50113,51.43,23.68,0.10,333,197121,1858,0.94
8,Link,l1Cancel,20240201 to 20240220,68369,556861,340914,61.22,18.06,0.11,307,865326,8763,1.01
9,Link,l2Cancel,20240201 to 20240220,66503,548554,333042,60.71,18.21,0.12,297,845452,8602,1.02


In [34]:
df_post_service

,service,pageVariant,date_range,customers,gross_orders,net_orders,g2n,ocara,aborted,median_ocara_time,view,click,ctr
0,Auto,l1Cancel,20240301 to 20240301,47636,77363,45593,58.93,19.01,0.12,285,146602,2026,1.38
1,Auto,l2Cancel,20240301 to 20240301,46929,75599,45385,60.03,18.30,0.14,302,146534,2013,1.37
2,Bike Lite,l1Cancel,20240301 to 20240301,14408,21131,14665,69.40,18.23,0.09,337,41227,521,1.26
3,Bike Lite,l2Cancel,20240301 to 20240301,13852,20455,14208,69.46,18.23,0.08,357,39801,535,1.34
4,Bike Metro,l1Cancel,20240301 to 20240301,1575,2006,1353,67.45,21.14,0.15,408,4805,47,0.98
5,Bike Metro,l2Cancel,20240301 to 20240301,1458,1917,1299,67.76,20.03,0.10,292,4327,61,1.41
6,CabEconomy,l1Cancel,20240301 to 20240301,15733,22871,13307,58.18,27.41,0.20,367,50030,639,1.28
7,CabEconomy,l2Cancel,20240301 to 20240301,15395,21993,13106,59.59,25.95,0.25,368,49010,621,1.27
8,Link,l1Cancel,20240301 to 20240301,65920,102520,64752,63.16,21.74,0.17,352,181679,2912,1.60
9,Link,l2Cancel,20240301 to 20240301,65290,100717,64624,64.16,20.87,0.19,379,182031,2850,1.57
